# Tune-up Experiments for Single Qubit

## Description

This is one of four template notebooks that together replace the original
monolithic workflow. **This notebook covers Tune-up
experiments** for a single qubit using the LabOne Q workflow architecture:

1. [Resonator Spectroscopy](#resonator-spectroscopy)
2. [Qubit Spectroscopy](#qubit-spectroscopy)
3. [Amplitude Rabi](#amplitude-rabi-ge) ([2nd](#amplitude-rabi-ef))
4. [Amplitude Rabi Chevron](#amplitude-rabi-chevron-ge) ([2nd](#amplitude-rabi-chevron-ef))
5. [Rabi Chevron](#rabi-chevron-ge) ([2nd](#rabi-chevron-ef))
6. [Rabi Frequency Calibration](#rabi-frequency-calibration-ge) ([2nd](#rabi-frequency-calibration-ef))
7. [T1 Lifetime](#t1-lifetime-ge) ([2nd](#t1-lifetime-ef))
8. [Ramsey](#ramsey-ge) ([2nd](#ramsey-ef))
9. [Rabi Error Amplification](#rabi-error-amplification-ge) ([2nd](#rabi-error-amplification-ef))
10. [DRAG Calibration](#drag-calibration-ge) ([2nd](#drag-calibration-ef))
11. [Hahn-Echo](#hahn-echo-ge) ([2nd](#hahn-echo-ef))
12. [Readout Optimisation](#readout-optimisation)

Run this notebook first for a fresh cooldown/qubit: it builds the `DeviceSetup`
and `QPU` from scratch and saves the tuned QPU to disk. The other three
template notebooks under `qubit_thermometry/templates/` load that saved QPU
and continue from there:

- Population and temperature measurements (single-shot 0/1, population sweeps).
- Fast Flux Drive.
- SINIS Calibration, Statistic and Temperature Sweep measurements.


## Changelog

All changes must be logged here! Do not make changes to this notebook other than expanding or modifying the standard experiments in a meaningful! All device-specific parameters and measurements have to be done in a different notebook!

Stick to semantic versioning:
- **Last Digit**: Fix a bug that does not add any new features and is backwards compatible -> increment last digit by one (e.g. 0.0.4 to 0.0.5)

- **Middle Digit**: Add a new feature while keeping backwards compatibility (older versions can still be executed without any modifications) -> increment middle digit by one and set last digit to 0 (e.g. 1.2.12 to 1.3.0)

- **First Digit**: Make changes that will break older versions, making them incompatible with the new version -> increment first digit by one and set all other digits to 0 (e.g. 3.2.9 to 4.0.0)

**Current Version:** <font color="#008000">**v2.0.0**</font>

**2026-09-27**, *Kha*: Updating from version v1.7.3, separated specified script for different experiments. Completed tune-up experiment. (v2.0.0)

# Package Imports and Initialization

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("C:/Users/phys-pico-lab/Documents/laboneq_workflow_notebooks/templates/qubit_thermometry_workflow-v2.0.0")
sys.path.insert(0, str(REPO_ROOT / "qubit_thermometry" / "src"))

In [ ]:
%load_ext autoreload
%autoreload 2

## LabOne Q and `laboneq_applications` Imports

In [ ]:
from laboneq.dsl.quantum import QPU
from laboneq_applications.qpu_types.tunable_transmon import (
    TunableTransmonQubit,
    TunableTransmonOperations,
)

from laboneq_applications.experiments import (
    resonator_spectroscopy,
    qubit_spectroscopy,
    amplitude_rabi,
    amplitude_fine,
    ramsey,
    drag_q_scaling,
    lifetime_measurement,
    echo,
    dispersive_shift,
)
from laboneq_applications.contrib.experiments import (
    amplitude_rabi_chevron,
)

## Other Imports

In [ ]:
from copy import deepcopy
from functools import partial
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt
from uncertainties import unumpy as unp

## Helpers and Shared Package

In [ ]:
from qubit_thermometry.helper import setup, qpu_io, data_io
from qubit_thermometry.workflow.experiments import rabi_chevron, rabi_frequency_calibration
from qubit_thermometry.workflow.analysis import (
    rabi_chevron as rabi_chevron_analysis,
    rabi_frequency_calibration as rabi_frequency_calibration_analysis,
    dispersive_shift as dispersive_shift_analysis,
)

## Descriptor and Device Setup

In [ ]:
number_of_qubits = 1
device_id = "" # TODO: placeholder: replace with instrument's device ID

assert device_id != "", 'Enter device ID'

device_setup, qubits = setup.build_device_setup(
    shfqc_device_id=device_id, 
    number_of_qubits=number_of_qubits,
)

## Qubit Initialization

In [ ]:
# TODO
cooldown_start_date = ''  # YYYY-MM-DD
sample_name = '' # Name of sample
qubit_name = '' # Qubit Q_N

assert cooldown_start_date != '', 'Enter cooldown name'
assert sample_name != '', 'Enter sample name'
assert qubit_name != '', "Enter qubit name"

In [ ]:
qubits  # Check in

In [ ]:
for q in qubits:
    q.parameters.ge_drive_pulse["sigma"] = 0.25
    q.parameters.readout_amplitude = 0.2
    q.parameters.reset_delay_length = 150e-6
    q.parameters.readout_range_out = -20
    q.parameters.readout_lo_frequency = 6e9
    q.parameters.drive_range = 0
    q.parameters.readout_range_in = -40
    q.parameters.readout_integration_delay = 80e-9

qubits[0].parameters.drive_lo_frequency = 4.8e9
qubits[0].parameters.resonance_frequency_ge = 4.85e9
qubits[0].parameters.resonance_frequency_ef = 4.65e9
qubits[0].parameters.readout_resonator_frequency = 6.003080000000001e9
qubits[0].parameters.spectroscopy_length = 10e-6
qubits[0].parameters.spectroscopy_amplitude = 0.2
qubits[0].uid = qubit_name

In [ ]:
qops = TunableTransmonOperations()
qpu = QPU(qubits, quantum_operations=qops)

tuning_log = []  # short strings appended after each "Update Parameters" cell

In [ ]:
for q in qubits:
    print("-------------")
    print("Qubit UID:", q.uid)
    print("Qubit logical signals:")
    for sig, lsg in q.signals.items():
        print(f"  {sig:<10} ('{lsg:>10}')")

## LabOne Q Session Connection

In [ ]:
session = setup.connect_session(device_setup, do_emulation=False)

## Data Saving

In [ ]:
data_root_directory, qpu_file_path = setup.init_data_saving(
    sample_name, cooldown_start_date, qubit_name
)
get_path_to_file = partial(setup.get_path_to_file, directory=data_root_directory)

In [ ]:
folder_store, logging_store = setup.setup_logbook(data_root_directory)

# Spectroscopy

## Resonator Spectroscopy <a class="anchor" id="resonator-spectroscopy"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

# TODO: Workflow parameters run #
readout_range_out = -25
readout_range_in = -30
drive_range = 5
readout_amplitude = 0.1
readout_resonator_frequency = 6.003e9
# ----------------------------- #

n_avg_exponent = 15

freq_range = 10e6
n_points = 501

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temp_pars.readout_range_out = readout_range_out
temp_pars.readout_range_in = readout_range_in
temp_pars.drive_range = drive_range
temp_pars.readout_amplitude = readout_amplitude
temp_pars.readout_resonator_frequency = readout_resonator_frequency
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = resonator_spectroscopy.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)

frequencies = temporary_parameters[qubit_to_measure.uid].readout_resonator_frequency + np.linspace(
    -freq_range / 2, freq_range / 2, n_points
)

exp_workflow = resonator_spectroscopy.experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    frequencies=frequencies,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

resonator_spectroscopy.update_qpu(qpu, qubit_parameters["new_parameter_values"])
print("\nReadout resonator frequency: ", qubit_to_measure.parameters.readout_resonator_frequency * 1e-9, ' GHz')

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append("Resonator Spectroscopy -> readout_resonator_frequency")
    print('\nUpdated global experiment parameters!\n')

## Qubit Spectroscopy <a class="anchor" id="qubit-spectroscopy"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

# TODO: Workflow parameters run #
# drive_range = 5
# readout_range_out = -20
# readout_range_in = -35
# ----------------------------- #

n_avg_exponent = 13

freq_range = 100e6
n_points = 251

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
# temp_pars.drive_range = drive_range
# temp_pars.readout_range_out = readout_range_out
# temp_pars.readout_range_in = readout_range_in
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = qubit_spectroscopy.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.update(False)
options.close_figures(False)

frequencies = [
    temporary_parameters[qubit_to_measure.uid].resonance_frequency_ge
    + np.linspace(-freq_range / 2, freq_range / 2, n_points)
]

exp_workflow = qubit_spectroscopy.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    frequencies=frequencies,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
# manual_center_frequency = 5.0191e9
# temporary_parameters[qubit_to_measure.uid].resonance_frequency_ge = manual_center_frequency

In [ ]:
update_global_parameters = False

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

qubit_spectroscopy.update_qpu(qpu, qubit_parameters["new_parameter_values"])
print("\nReadout resonance frequency ge: ", qubit_to_measure.parameters.resonance_frequency_ge)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append("Qubit Spectroscopy -> resonance_frequency_ge")
    print('\nUpdated global experiment parameters!\n')

# Control Pulses

## First Transition: g-e

### Amplitude Rabi <a class="anchor" id="amplitude-rabi-ge"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

# NOTE: `drive_length` was used but never explicitly defined in the original
# g-e Amplitude Rabi cell (it relied on stale kernel state left over from a
# previous e-f run). Defined explicitly here with the same default as the
# e-f block below.
# TODO: Workflow parameters run #
drive_length = 50e-9
# ----------------------------- #

n_avg_exponent = 13

amp_min = 0
amp_max = 0.5
n_amp = 41

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
if transition == 'ge':
    temp_pars.ge_drive_length = drive_length
elif transition == 'ef':
    temp_pars.ef_drive_length = drive_length
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = amplitude_rabi.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    amplitudes=[np.linspace(amp_min, amp_max, n_amp)],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

amplitude_rabi.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nDrive amplitude pi ge: ", qubit_to_measure.parameters.ge_drive_amplitude_pi)
    print("Drive amplitude pi2 ge: ", qubit_to_measure.parameters.ge_drive_amplitude_pi2)
elif transition == 'ef':
    print("\nDrive amplitude pi ef: ", qubit_to_measure.parameters.ef_drive_amplitude_pi)
    print("Drive amplitude pi2 ef: ", qubit_to_measure.parameters.ef_drive_amplitude_pi2)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Amplitude Rabi ({transition}) -> drive_amplitude_pi/pi2")
    print('\nUpdated global experiment parameters!\n')

### Amplitude Rabi Chevron <a class="anchor" id="amplitude-rabi-chevron-ge"></a>

#### Experiment Parameters

In [ ]:
transition = setup.set_transition('ge')

n_avg_exponent = 13

freq_range = 100e6
n_points = 251

amp_min = 0
amp_max = 1
n_amp = 41

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = amplitude_rabi_chevron.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

# NOTE: ported as-is from the original notebook, which always used
# `resonance_frequency_ge` here regardless of `transition` (even in the e-f
# mirror below). This looks like a copy-paste bug in the source notebook;
# flagged for review rather than silently changed.
frequencies = [
    temporary_parameters[qubit_to_measure.uid].resonance_frequency_ge
    + np.linspace(-freq_range / 2, freq_range / 2, n_points)
]

exp_workflow = amplitude_rabi_chevron.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    frequencies=frequencies,
    amplitudes=[np.linspace(amp_min, amp_max, n_amp)],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

### Rabi Chevron <a class="anchor" id="rabi-chevron-ge"></a>

#### Experiment Parameters

In [ ]:
transition = setup.set_transition('ge')

n_avg_exponent = 13

amp_min = 0
amp_max = 1
n_amp = 41
rabi_chevron_amplitudes = np.linspace(amp_min, amp_max, n_amp)

# frequency detuning range swept around the qubit's current resonance frequency
chevron_detuning_min = -6e6
chevron_detuning_max = 2e6
n_detuning = 9
chevron_detunings = np.linspace(chevron_detuning_min, chevron_detuning_max, n_detuning)

qubit_to_measure = qubits[0]

temporary_parameters_base = deepcopy(qubit_to_measure.parameters)

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)  # suppress per-detuning plots; one 2D map is built below

options.transition(transition)
options.cal_states(transition)

rabi_chevron_arr = rabi_chevron.run_rabi_chevron_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    transition=transition,
    amplitudes=rabi_chevron_amplitudes,
    detunings=chevron_detunings,
    options=options,
    folder_store=folder_store,
    logging_store=logging_store,
    temporary_parameters_base=temporary_parameters_base,
)

#### Plot and Save Data

In [ ]:
rabi_chevron_analysis.plot_chevron_map(
    rabi_chevron_amplitudes, chevron_detunings, rabi_chevron_arr, transition, get_path_to_file
)

payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date,
    data={
        'rabi_chevron': rabi_chevron_arr,
        'detunings': chevron_detunings,
        'rabi_amp': rabi_chevron_amplitudes,
    },
)
data_io.save_mat(get_path_to_file(f'Rabi_chevron_{transition}_', '.mat'), payload)

### Rabi Frequency Calibration <a class="anchor" id="rabi-frequency-calibration-ge"></a>

#### Experiment Parameters

In [ ]:
# Diagnostic/informational: derives a slope/intercept relation between
# drive-pulse length and the required pi-pulse amplitude. Does not directly
# overwrite qubit_to_measure.parameters below.

transition = setup.set_transition('ge')

n_avg_exponent = 12

amp_min = 0
amp_max = 1.0
n_amp = 101
rabi_freq_calib_amplitudes = np.linspace(amp_min, amp_max, n_amp)

drive_length_min = 50e-9
drive_length_max = 400e-9
n_drive_length = 36
drive_lengths = np.linspace(drive_length_min, drive_length_max, n_drive_length)

qubit_to_measure = qubits[0]

temporary_parameters_base = deepcopy(qubit_to_measure.parameters)

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)  # suppress per-length plots; a summary plot is built below

options.transition(transition)
options.cal_states(transition)

fitted_pi_amplitudes = rabi_frequency_calibration.run_rabi_frequency_calibration_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    transition=transition,
    amplitudes=rabi_freq_calib_amplitudes,
    drive_lengths=drive_lengths,
    options=options,
    folder_store=folder_store,
    logging_store=logging_store,
    temporary_parameters_base=temporary_parameters_base,
)

#### Fit, Plot, and Save Data

In [ ]:
rabi_slope, rabi_intercept, *_ = rabi_frequency_calibration_analysis.fit_and_plot_rabi_frequency_calibration(
    drive_lengths, fitted_pi_amplitudes, transition, get_path_to_file
)

payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date,
    data={
        'drive_lengths': drive_lengths,
        'fitted_pi_amplitudes': unp.nominal_values(fitted_pi_amplitudes).astype('float64'),
        'rabi_slope': rabi_slope,
        'rabi_intercept': rabi_intercept,
    },
)
data_io.save_mat(get_path_to_file(f'Rabi_freq_calib_{transition}_', '.mat'), payload)

### T1 Lifetime <a class="anchor" id="t1-lifetime-ge"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

n_avg_exponent = 13

delay_min = 0
delay_max = 20e-6
n_delay = 51
log_spacing = True

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = lifetime_measurement.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

if log_spacing:
    delays = [setup.log_sweep_help(delay_min, delay_max, n_delay)]
else:
    delays = [np.linspace(delay_min, delay_max, n_delay)]

exp_workflow = lifetime_measurement.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    delays=delays,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

lifetime_measurement.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_T1: ", qubit_to_measure.parameters.ge_T1)
elif transition == 'ef':
    print("\nef_T1: ", qubit_to_measure.parameters.ef_T1)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"T1 Lifetime ({transition}) -> {transition}_T1")
    print('\nUpdated global experiment parameters!\n')

In [ ]:
qubits[0].parameters.reset_delay_length = 40e-6
# qubits[0].parameters.readout_range_in   = -20
# qubits[0].parameters.readout_range_out  = -30

### Ramsey <a class="anchor" id="ramsey-ge"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

detuning = 5e6
transition = setup.set_transition('ge')

n_avg_exponent = 13

delay_min = 0
delay_max = 1e-6
n_delay = 101

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = ramsey.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

# Note: if neighbouring qubits are physically coupled by a resonator, you
# usually don't want to run Ramsey in parallel on them because the result
# will be skewed by strong residual-ZZ coupling. Next-nearest neighbours is
# typically okay.
delays = [np.linspace(delay_min, delay_max, n_delay)]
detunings = [detuning]

exp_workflow = ramsey.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    delays=delays,
    detunings=detunings,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

ramsey.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_T2_star: ", qubit_to_measure.parameters.ge_T2_star)
    print("resonance_frequency_ge: ", qubit_to_measure.parameters.resonance_frequency_ge)
elif transition == 'ef':
    print("\nef_T2_star: ", qubit_to_measure.parameters.ef_T2_star)
    print("resonance_frequency_ef: ", qubit_to_measure.parameters.resonance_frequency_ef)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Ramsey ({transition}) -> {transition}_T2_star, resonance_frequency_{transition}")
    print('\nUpdated global experiment parameters!\n')

### Rabi Error Amplification <a class="anchor" id="rabi-error-amplification-ge"></a>

#### x90 pulse

##### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

n_avg_exponent = 13

x90_max_repetitions = 15
x90_repetitions = list(range(2, x90_max_repetitions + 1, 2))

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

##### Run Workflow

In [ ]:
options = amplitude_fine.experiment_workflow_x90.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = amplitude_fine.experiment_workflow_x90(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    repetitions=[x90_repetitions],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

##### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

amplitude_fine.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nDrive amplitude pi2 ge (error-amplified): ", qubit_to_measure.parameters.ge_drive_amplitude_pi2)
elif transition == 'ef':
    print("\nDrive amplitude pi2 ef (error-amplified): ", qubit_to_measure.parameters.ef_drive_amplitude_pi2)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Rabi Error Amplification x90 ({transition}) -> {transition}_drive_amplitude_pi2")
    print('\nUpdated global experiment parameters!\n')

#### x180 pulse

##### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

n_avg_exponent = 13

x180_max_repetitions = 16
x180_repetitions = list(range(0, x180_max_repetitions + 1))

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

##### Run Workflow

In [ ]:
options = amplitude_fine.experiment_workflow_x180.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = amplitude_fine.experiment_workflow_x180(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    repetitions=[x180_repetitions],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

##### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

amplitude_fine.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nDrive amplitude pi ge (error-amplified): ", qubit_to_measure.parameters.ge_drive_amplitude_pi)
elif transition == 'ef':
    print("\nDrive amplitude pi ef (error-amplified): ", qubit_to_measure.parameters.ef_drive_amplitude_pi)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Rabi Error Amplification x180 ({transition}) -> {transition}_drive_amplitude_pi")
    print('\nUpdated global experiment parameters!\n')

### DRAG Calibration <a class="anchor" id="drag-calibration-ge"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

n_avg_exponent = 13

q_min = -0.1
q_max = 0.1
n_q = 61

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = drag_q_scaling.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

# NOTE: the original g-e cell does not pass `temporary_parameters` here
# (unlike its e-f mirror below), so `temp_pars` above has no effect for this
# transition. Preserved as-is from the source notebook.
exp_workflow = drag_q_scaling.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    q_scalings=[np.linspace(q_min, q_max, n_q)],
    options=options,
    temparary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

drag_q_scaling.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_drive_pulse[beta]: ", qubit_to_measure.parameters.ge_drive_pulse["beta"])
elif transition == 'ef':
    print("\nef_drive_pulse[beta]: ", qubit_to_measure.parameters.ef_drive_pulse["beta"])

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"DRAG Calibration ({transition}) -> {transition}_drive_pulse[beta]")
    print('\nUpdated global experiment parameters!\n')

### Hahn-Echo <a class="anchor" id="hahn-echo-ge"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

n_avg_exponent = 13

delay_min = 0
delay_max = 50e-6
n_delay = 51
log_spacing = True

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = echo.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)

options.transition(transition)
options.cal_states(transition)

# Note: if neighbouring qubits are physically coupled by a resonator, you
# usually don't want to run Echo in parallel on them because the result will
# be skewed by strong residual-ZZ coupling. Next-nearest neighbours is
# typically okay.
delays = [np.linspace(delay_min, delay_max, n_delay)]

exp_workflow = echo.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    delays=delays,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

echo.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_T2: ", qubit_to_measure.parameters.ge_T2)
elif transition == 'ef':
    print("\nef_T2: ", qubit_to_measure.parameters.ef_T2)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Hahn-Echo ({transition}) -> {transition}_T2")
    print('\nUpdated global experiment parameters!\n')

## Second Transition: e-f

### Amplitude Rabi <a class="anchor" id="amplitude-rabi-ef"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = True

transition = setup.set_transition('ef')

# TODO: Workflow parameters run #
drive_length = 50e-9
# ----------------------------- #

n_avg_exponent = 13

amp_min = 0
amp_max = 0.5
n_amp = 41

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
if transition == 'ge':
    temp_pars.ge_drive_length = drive_length
elif transition == 'ef':
    temp_pars.ef_drive_length = drive_length
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = amplitude_rabi.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    amplitudes=[np.linspace(amp_min, amp_max, n_amp)],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

amplitude_rabi.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nDrive amplitude pi ge: ", qubit_to_measure.parameters.ge_drive_amplitude_pi)
    print("Drive amplitude pi2 ge: ", qubit_to_measure.parameters.ge_drive_amplitude_pi2)
elif transition == 'ef':
    print("\nDrive amplitude pi ef: ", qubit_to_measure.parameters.ef_drive_amplitude_pi)
    print("Drive amplitude pi2 ef: ", qubit_to_measure.parameters.ef_drive_amplitude_pi2)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Amplitude Rabi ({transition}) -> drive_amplitude_pi/pi2")
    print('\nUpdated global experiment parameters!\n')

### Amplitude Rabi Chevron <a class="anchor" id="amplitude-rabi-chevron-ef"></a>

#### Experiment Parameters

In [ ]:
transition = setup.set_transition('ef')

n_avg_exponent = 13

freq_range = 100e6
n_points = 251

amp_min = 0
amp_max = 1
n_amp = 41

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = amplitude_rabi_chevron.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

# NOTE: ported as-is from the original notebook, which always used
# `resonance_frequency_ge` here regardless of `transition`. This looks like
# a copy-paste bug in the source notebook; flagged for review rather than
# silently changed.
frequencies = [
    temporary_parameters[qubit_to_measure.uid].resonance_frequency_ef
    + np.linspace(-freq_range / 2, freq_range / 2, n_points)
]

exp_workflow = amplitude_rabi_chevron.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    frequencies=frequencies,
    amplitudes=[np.linspace(amp_min, amp_max, n_amp)],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

### Rabi Chevron <a class="anchor" id="rabi-chevron-ef"></a>

#### Experiment Parameters

In [ ]:
transition = setup.set_transition('ef')

n_avg_exponent = 12

amp_min = 0
amp_max = 1
n_amp = 61
rabi_chevron_amplitudes = np.linspace(amp_min, amp_max, n_amp)

chevron_detuning_min = -10.0e6
chevron_detuning_max = 10.0e6
n_detuning = 21
chevron_detunings = np.linspace(chevron_detuning_min, chevron_detuning_max, n_detuning)

qubit_to_measure = qubits[0]

temporary_parameters_base = deepcopy(qubit_to_measure.parameters)

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)  # suppress per-detuning plots; one 2D map is built below

options.transition(transition)
options.cal_states(transition)

rabi_chevron_arr = rabi_chevron.run_rabi_chevron_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    transition=transition,
    amplitudes=rabi_chevron_amplitudes,
    detunings=chevron_detunings,
    options=options,
    folder_store=folder_store,
    logging_store=logging_store,
    temporary_parameters_base=temporary_parameters_base,
)

#### Plot and Save Data

In [ ]:
rabi_chevron_analysis.plot_chevron_map(
    rabi_chevron_amplitudes, chevron_detunings, rabi_chevron_arr, transition, get_path_to_file
)

payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date,
    data={
        'rabi_chevron': rabi_chevron_arr,
        'detunings': chevron_detunings,
        'rabi_amp': rabi_chevron_amplitudes,
    },
)
data_io.save_mat(get_path_to_file(f'Rabi_chevron_{transition}_', '.mat'), payload)

### Rabi Frequency Calibration <a class="anchor" id="rabi-frequency-calibration-ef"></a>

#### Experiment Parameters

In [ ]:
transition = setup.set_transition('ef')

n_avg_exponent = 12

amp_min = 0
amp_max = 1.0
n_amp = 101
rabi_freq_calib_amplitudes = np.linspace(amp_min, amp_max, n_amp)

drive_length_min = 50e-9
drive_length_max = 400e-9
n_drive_length = 36
drive_lengths = np.linspace(drive_length_min, drive_length_max, n_drive_length)

qubit_to_measure = qubits[0]

temporary_parameters_base = deepcopy(qubit_to_measure.parameters)

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)  # suppress per-length plots; a summary plot is built below

options.transition(transition)
options.cal_states(transition)

fitted_pi_amplitudes = rabi_frequency_calibration.run_rabi_frequency_calibration_sweep(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure,
    transition=transition,
    amplitudes=rabi_freq_calib_amplitudes,
    drive_lengths=drive_lengths,
    options=options,
    folder_store=folder_store,
    logging_store=logging_store,
    temporary_parameters_base=temporary_parameters_base,
)

#### Fit, Plot, and Save Data

In [ ]:
rabi_slope, rabi_intercept, *_ = rabi_frequency_calibration_analysis.fit_and_plot_rabi_frequency_calibration(
    drive_lengths, fitted_pi_amplitudes, transition, get_path_to_file
)

payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date,
    data={
        'drive_lengths': drive_lengths,
        'fitted_pi_amplitudes': unp.nominal_values(fitted_pi_amplitudes).astype('float64'),
        'rabi_slope': rabi_slope,
        'rabi_intercept': rabi_intercept,
    },
)
data_io.save_mat(get_path_to_file(f'Rabi_freq_calib_{transition}_', '.mat'), payload)

### T1 Lifetime <a class="anchor" id="t1-lifetime-ef"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ef')

n_avg_exponent = 13

delay_min = 0
delay_max = 20e-6
n_delay = 71
log_spacing = True

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = lifetime_measurement.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

if log_spacing:
    delays = [setup.log_sweep_help(delay_min, delay_max, n_delay)]
else:
    delays = [np.linspace(delay_min, delay_max, n_delay)]

exp_workflow = lifetime_measurement.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    delays=delays,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

lifetime_measurement.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_T1: ", qubit_to_measure.parameters.ge_T1)
elif transition == 'ef':
    print("\nef_T1: ", qubit_to_measure.parameters.ef_T1)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"T1 Lifetime ({transition}) -> {transition}_T1")
    print('\nUpdated global experiment parameters!\n')

### Ramsey <a class="anchor" id="ramsey-ef"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

detuning = 5e6
transition = setup.set_transition('ef')

n_avg_exponent = 13

delay_min = 0
delay_max = 1e-6
n_delay = 101

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = ramsey.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

# Note: if neighbouring qubits are physically coupled by a resonator, you
# usually don't want to run Ramsey in parallel on them because the result
# will be skewed by strong residual-ZZ coupling. Next-nearest neighbours is
# typically okay.
delays = [np.linspace(delay_min, delay_max, n_delay)]
detunings = [detuning]

exp_workflow = ramsey.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    delays=delays,
    detunings=detunings,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

ramsey.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_T2_star: ", qubit_to_measure.parameters.ge_T2_star)
    print("resonance_frequency_ge: ", qubit_to_measure.parameters.resonance_frequency_ge)
elif transition == 'ef':
    print("\nef_T2_star: ", qubit_to_measure.parameters.ef_T2_star)
    print("resonance_frequency_ef: ", qubit_to_measure.parameters.resonance_frequency_ef)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Ramsey ({transition}) -> {transition}_T2_star, resonance_frequency_{transition}")
    print('\nUpdated global experiment parameters!\n')

### Rabi Error Amplification <a class="anchor" id="rabi-error-amplification-ef"></a>

#### x180 pulse

##### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ef')

n_avg_exponent = 13

x180_max_repetitions = 16
x180_repetitions = list(range(0, x180_max_repetitions + 1))

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

##### Run Workflow

In [ ]:
options = amplitude_fine.experiment_workflow_x180.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = amplitude_fine.experiment_workflow_x180(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    repetitions=[x180_repetitions],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

##### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

amplitude_fine.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nDrive amplitude pi ge (error-amplified): ", qubit_to_measure.parameters.ge_drive_amplitude_pi)
elif transition == 'ef':
    print("\nDrive amplitude pi ef (error-amplified): ", qubit_to_measure.parameters.ef_drive_amplitude_pi)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Rabi Error Amplification x180 ({transition}) -> {transition}_drive_amplitude_pi")
    print('\nUpdated global experiment parameters!\n')

#### x90 pulse

##### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ef')

n_avg_exponent = 13

x90_max_repetitions = 16
x90_repetitions = list(range(0, x90_max_repetitions + 1, 2))

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

##### Run Workflow

In [ ]:
options = amplitude_fine.experiment_workflow_x90.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = amplitude_fine.experiment_workflow_x90(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    repetitions=[x90_repetitions],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

##### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

amplitude_fine.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nDrive amplitude pi2 ge (error-amplified): ", qubit_to_measure.parameters.ge_drive_amplitude_pi2)
elif transition == 'ef':
    print("\nDrive amplitude pi2 ef (error-amplified): ", qubit_to_measure.parameters.ef_drive_amplitude_pi2)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Rabi Error Amplification x90 ({transition}) -> {transition}_drive_amplitude_pi2")
    print('\nUpdated global experiment parameters!\n')

### DRAG Calibration <a class="anchor" id="drag-calibration-ef"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ef')

n_avg_exponent = 13

q_min = -0.1
q_max = 0.1
n_q = 61

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = drag_q_scaling.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)  # Plot figures

options.transition(transition)
options.cal_states(transition)

exp_workflow = drag_q_scaling.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    q_scalings=[np.linspace(q_min, q_max, n_q)],
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

drag_q_scaling.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_drive_pulse[beta]: ", qubit_to_measure.parameters.ge_drive_pulse["beta"])
elif transition == 'ef':
    print("\nef_drive_pulse[beta]: ", qubit_to_measure.parameters.ef_drive_pulse["beta"])

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"DRAG Calibration ({transition}) -> {transition}_drive_pulse[beta]")
    print('\nUpdated global experiment parameters!\n')

### Hahn-Echo <a class="anchor" id="hahn-echo-ef"></a>

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ef')

n_avg_exponent = 13

delay_min = 0
delay_max = 15e-6
n_delay = 51
log_spacing = True

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = echo.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)

options.transition(transition)
options.cal_states(transition)

# Note: if neighbouring qubits are physically coupled by a resonator, you
# usually don't want to run Echo in parallel on them because the result will
# be skewed by strong residual-ZZ coupling. Next-nearest neighbours is
# typically okay.
delays = [np.linspace(delay_min, delay_max, n_delay)]

exp_workflow = echo.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    delays=delays,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks["analysis_workflow"]
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])

echo.update_qpu(qpu, qubit_parameters["new_parameter_values"])
if transition == 'ge':
    print("\nge_T2: ", qubit_to_measure.parameters.ge_T2)
elif transition == 'ef':
    print("\nef_T2: ", qubit_to_measure.parameters.ef_T2)

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append(f"Hahn-Echo ({transition}) -> {transition}_T2")
    print('\nUpdated global experiment parameters!\n')

# Readout Optimisation <a class="anchor" id="readout-optimisation"></a>

Optimization of the readout pulse amplitude and of the readout (resonator)
frequency.

1. **Readout Amplitude Optimization** - sweep the readout amplitude and run an
   amplitude-Rabi experiment at each amplitude. The readout amplitude with the
   smallest relative Rabi-fit errors gives the best signal-to-noise ratio.
2. **Dispersive Shift** - resonator spectroscopy with the qubit prepared in
   g, e (and f). The optimal readout frequency is the frequency of the largest
   distance between the transmission signals of the different states.
3. **Readout Settings Summary** - print the optimized readout parameters
   (already written into the QPU and saved by the two steps above).


## Readout Amplitude Optimisation

#### Experiment Parameters

In [ ]:
update_global_parameters = False

transition = setup.set_transition('ge')

n_avg_exponent = 13

# TODO: Workflow parameters run #
# set these parameters when running readout amp opt before
# running tune-up experiments,
# readout setting used during the sweep
readout_range_out = -25
readout_range_in = -30  # -40 TWPA off, -30 TWPA on
readout_length = 2e-6
readout_integration_length = 2e-6
readout_integration_delay = 80e-9
reset_delay_length = 40e-6

# length and amplitude are taken from readout_length / readout_amplitude
readout_pulse = {
    'function': 'gaussian_square',
    'width': readout_length * 0.95,
}
# ----------------------------- #

# readout amplitude sweep
ro_amp_min = 0.0
ro_amp_max = 1.0
n_ro_amp = 15
ro_amp_arr = np.linspace(ro_amp_min, ro_amp_max, n_ro_amp)

# rabi sweep used to benchmark each readout amplitude
drive_range = 0
amp_min = 0
amp_max = 1
n_amp = 31
drive_amplitudes = np.linspace(amp_min, amp_max, n_amp)

qubit_to_measure = qubits[0]

#### Run Workflow

In [ ]:
options = amplitude_rabi.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)  # one Rabi figure per readout amplitude

options.transition(transition)
options.cal_states(transition)

fit_values_ro_sweep = []
fit_errors_ro_sweep = []
# same order as the old popt/pcov of fit_Rabi
fit_parameter_names = ['frequency', 'phase', 'amplitude', 'offset']

with setup.logging_disabled(folder_store, logging_store):
    for ro_amp in ro_amp_arr:
        print('Measure at readout amplitude:', round(ro_amp, 5))

        temp_pars = deepcopy(qubit_to_measure.parameters)
        temp_pars.readout_amplitude = round(ro_amp, 5)
        temp_pars.readout_length = readout_length
        temp_pars.readout_pulse = readout_pulse
        temp_pars.readout_integration_length = readout_integration_length
        temp_pars.readout_integration_delay = readout_integration_delay
        temp_pars.readout_range_out = readout_range_out
        temp_pars.readout_range_in = readout_range_in
        temp_pars.reset_delay_length = reset_delay_length
        temp_pars.drive_range = drive_range
        temporary_parameters = {qubit_to_measure.uid: temp_pars}

        exp_workflow = amplitude_rabi.experiment_workflow(
            session=session,
            qpu=qpu,
            qubits=[qubit_to_measure.uid],
            amplitudes=[drive_amplitudes],
            options=options,
            temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()

        # rotates/projects the raw data and fits a cosine model to the population
        fit_results = setup.get_analysis_task_output(workflow_result, 'fit_data')
        fit_res = fit_results.get(qubit_to_measure.uid)
        if fit_res is None:
            print(f'  Rabi fit failed at readout amplitude {round(ro_amp, 5)}.')
            fit_values_ro_sweep.append([np.nan] * len(fit_parameter_names))
            fit_errors_ro_sweep.append([np.nan] * len(fit_parameter_names))
            continue

        fit_values_ro_sweep.append([fit_res.params[p].value for p in fit_parameter_names])
        fit_errors_ro_sweep.append([fit_res.params[p].stderr for p in fit_parameter_names])

popt_rabi_arr = np.array(fit_values_ro_sweep, dtype=float)
err_arr = np.array(fit_errors_ro_sweep, dtype=float)

#### Update Parameters

In [ ]:
update_global_parameters = True
ll = 1  # skip readout amplitude 0 (no signal)
ul = None
sweep_slice = slice(ll, ul)

# lmfit reports stderr = 0 when the covariance matrix could not be estimated;
# treat those points as invalid instead of "perfect".
err_arr[err_arr == 0] = np.nan
rel_err_arr = np.abs(err_arr / popt_rabi_arr)
fig, ax = plt.subplots(2, 2, sharex=True, figsize=(10, 8))
fig.suptitle(f'Relative errors for Rabi measurements - {qubit_to_measure.uid}', fontsize=16)
fig.supylabel('Relative error (a.u.)')

ax[0, 0].plot(ro_amp_arr[sweep_slice], rel_err_arr[sweep_slice, 0], '.k', label='freq')
ax[1, 0].plot(ro_amp_arr[sweep_slice], rel_err_arr[sweep_slice, 1], '.k', label='phase')
ax[0, 1].plot(ro_amp_arr[sweep_slice], rel_err_arr[sweep_slice, 2], '.k', label='amp')
ax[1, 1].plot(ro_amp_arr[sweep_slice], rel_err_arr[sweep_slice, 3], '.k', label='off')

ax[0, 0].set_title('Frequency')
ax[1, 0].set_title('Phase')
ax[0, 1].set_title('Amplitude')
ax[1, 1].set_title('Offset')

# optimal readout amplitude: smallest mean relative fit error.
mean_rel_err = np.nanmean(rel_err_arr, axis=1)
masked_rel_err = np.full(mean_rel_err.shape, np.nan)
masked_rel_err[sweep_slice] = mean_rel_err[sweep_slice]
optimal_index = int(np.nanargmin(masked_rel_err))
readout_amplitude_opt = float(ro_amp_arr[optimal_index])

for axis in ax.flatten():
    axis.axvline(x=readout_amplitude_opt, ls='-.', color='r', label='optimal')
    axis.legend()

print('Optimal readout amplitude:', round(readout_amplitude_opt, 5))

file_path = get_path_to_file(f'Rabi_optimiz_{readout_range_out}dBm_', '.png')
fig.savefig(file_path, dpi=600, format='png', bbox_inches='tight')

In [ ]:
# set to a value != None to override the value extracted above.
manual_readout_amplitude = 0.6
if manual_readout_amplitude is not None:
    readout_amplitude_opt = float(manual_readout_amplitude)

if update_global_parameters:
    temp_pars.readout_amplitude = readout_amplitude_opt
    qubit_to_measure.parameters = deepcopy(temp_pars)
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append('Readout Amplitude Optimisation -> readout_amplitude')
    print('\nUpdated global experiment parameters!\n')

print('Readout amplitude: ', qubit_to_measure.parameters.readout_amplitude)
print('Readout length: ', qubit_to_measure.parameters.readout_length)
print('Readout pulse: ', qubit_to_measure.parameters.readout_pulse)
print('Readout integration length: ', qubit_to_measure.parameters.readout_integration_length)
print('Readout integration delay: ', qubit_to_measure.parameters.readout_integration_delay)
print('Readout range out / in: ', qubit_to_measure.parameters.readout_range_out,
      '/', qubit_to_measure.parameters.readout_range_in)

## Dispersive Shift

#### Experiment Parameters

In [ ]:
update_global_parameters = False

states = 'gef'

n_avg_exponent = 13

freq_range = 6e6
n_points = 201

qubit_to_measure = qubits[0]

temporary_parameters = {}
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters[qubit_to_measure.uid] = temp_pars

#### Run Workflow

In [ ]:
options = dispersive_shift.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)

frequencies = temporary_parameters[qubit_to_measure.uid].readout_resonator_frequency + np.linspace(
    -freq_range / 2, freq_range / 2, n_points
)

exp_workflow = dispersive_shift.experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    frequencies=frequencies,
    states=states,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Analysis Plots

In [ ]:
dispersive_shift_results = dispersive_shift_analysis.plot_dispersive_shift_results(
    workflow_result, qubit_to_measure, frequencies, states, get_path_to_file
)

#### Update Parameters

In [ ]:
update_global_parameters = True

analysis_workflow_result = workflow_result.tasks['analysis_workflow']
qubit_parameters = analysis_workflow_result.output
pprint(qubit_parameters)

readout_resonator_frequency_opt = float(
    qubit_parameters['new_parameter_values'][qubit_to_measure.uid]['readout_resonator_frequency']
)

manual_readout_resonator_frequency = None
if manual_readout_resonator_frequency is not None:
    readout_resonator_frequency_opt = float(manual_readout_resonator_frequency)
    qubit_parameters['new_parameter_values'][qubit_to_measure.uid][
        'readout_resonator_frequency'
    ] = readout_resonator_frequency_opt

print(f'ro freq opt: {readout_resonator_frequency_opt * 1e-9:.6f} GHz')

if update_global_parameters:
    temporary_parameters[qubit_to_measure.uid].readout_resonator_frequency = readout_resonator_frequency_opt
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])
    dispersive_shift.update_qpu(qpu, qubit_parameters['new_parameter_values'])

print('Readout resonator frequency: ', qubit_to_measure.parameters.readout_resonator_frequency * 1e-9, ' GHz')

if update_global_parameters:
    qpu_io.save_qpu(qpu, qpu_file_path)
    tuning_log.append('Dispersive Shift -> readout_resonator_frequency')
    print('\nUpdated global experiment parameters!\n')

In [ ]:
data_gef_spec = {'freq': frequencies}
for state, data in dispersive_shift_results['res_data'].items():
    data_gef_spec[f'{state}_res'] = data
    data_gef_spec[f'{state}_freq'] = frequencies

payload = data_io.build_mat_payload(
    qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_gef_spec
)
data_io.save_mat(get_path_to_file('ge_spec_readout_', '.mat'), payload)

## Readout Settings Summary

In [ ]:
readout_opt = {
    'readout_amplitude': qubit_to_measure.parameters.readout_amplitude,
    'readout_length': qubit_to_measure.parameters.readout_length,
    'readout_pulse': qubit_to_measure.parameters.readout_pulse,
    'readout_integration_length': qubit_to_measure.parameters.readout_integration_length,
    'readout_integration_delay': qubit_to_measure.parameters.readout_integration_delay,
    'readout_integration_kernels_type': qubit_to_measure.parameters.readout_integration_kernels_type,
    'readout_resonator_frequency': qubit_to_measure.parameters.readout_resonator_frequency,
    'readout_lo_frequency': qubit_to_measure.parameters.readout_lo_frequency,
    'readout_range_out': qubit_to_measure.parameters.readout_range_out,
    'readout_range_in': qubit_to_measure.parameters.readout_range_in,
    'reset_delay_length': qubit_to_measure.parameters.reset_delay_length,
}
pprint(readout_opt)

# Summary

In [ ]:
qpu_io.print_qpu_summary(qpu, qpu_file_path, tuning_log)